# Graph Classification using GCN, GAT, and GraphSAGE

This notebook implements and evaluates three Graph Neural Network models — GCN, GAT, and GraphSAGE — for node classification on a custom graph dataset loaded from `edges.csv`, `features.csv`, and `labels.csv`.

Metrics reported: **Precision** and **Recall** (macro-averaged).

## Step 1: Install Required Libraries

In [15]:
# # Install PyTorch Geometric and its dependencies
# import subprocess, sys, torch

# torch_ver = torch.__version__
# print(f"PyTorch version: {torch_ver}")

# # Install torch_geometric (compatible with current torch)
# # subprocess.check_call([sys.executable, "-m", "pip", "install", "torch_geometric", "-q"])
# !pip install torch_geometric -q
# # Install optional dependencies (scatter, sparse, cluster, spline_conv)
# # try:
# # except ImportEtfarror:
#     # subprocess.check_call([sys.executable, "-m", "pip", "install",
#     #     "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv",
#     #     "-f", f"https://data.pyg.org/whl/torch-{torch_ver}.html", "-q"])
# !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv
# import torch_scatter

# print("All libraries installed successfully.")
# Detect torch & CUDA versions
import torch
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda.replace('.', '')

print("Torch:", TORCH)
print("CUDA:", CUDA)

# Install PyG compiled wheels (no source build)
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-{TORCH}+cu{CUDA}.html

# Install main library
!pip install -q torch_geometric

print("PyTorch Geometric installed successfully!")

Torch: 2.10.0
CUDA: 128
ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib
PyTorch Geometric installed successfully!


## Step 2: Import Libraries

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from sklearn.metrics import precision_score, recall_score, classification_report

print(f"PyTorch: {torch.__version__}")
import torch_geometric
print(f"PyTorch Geometric: {torch_geometric.__version__}")


PyTorch: 2.10.0+cu128
PyTorch Geometric: 2.7.0


## Step 3: Load the Dataset CSVs

Upload `edges.csv`, `features.csv`, and `labels.csv` to `/content/` before running this cell.

> **File format expected:**
> - `edges.csv` — columns: `Vertex 1`, `Vertex 2`
> - `features.csv` — columns: `Vertex`, `f1` … `f16`
> - `labels.csv` — columns: `Vertex`, `label`

In [17]:
# ── Load CSV files ──────────────────────────────────────────────────────────
edges_df   = pd.read_csv('/content/edges.csv')
features_df = pd.read_csv('/content/features.csv')
labels_df  = pd.read_csv('/content/labels.csv')
features_df = features_df.rename(columns={'Vertex ': 'Vertex'})

print("=== edges.csv ===");  print(edges_df.head()); print(edges_df.shape)
print("\n=== features.csv ==="); print(features_df.head()); print(features_df.shape)
print("\n=== labels.csv ===");  print(labels_df.head()); print(labels_df.shape)


=== edges.csv ===
   Vertex 1  Vertex 2
0      1006      1061
1       101       689
2      1255      1765
3       532      1141
4      1425      1947
(8000, 2)

=== features.csv ===
   Vertex        f1        f2        f3        f4        f5        f6  \
0       0  0.749816  0.980286  0.892798  0.839463  0.662407  0.662398   
1       1  0.721697  0.809903  0.772778  0.716492  0.844741  0.655798   
2       2  0.626021  0.979554  0.986253  0.923359  0.721846  0.639069   
3       3  0.818684  0.673942  0.987834  0.910053  0.975800  0.957931   
4       4  0.712374  0.817078  0.656370  0.920879  0.629820  0.994755   

         f7        f8        f9       f10       f11       f12       f13  \
0  0.623233  0.946470  0.840446  0.883229  0.608234  0.987964  0.932977   
1  0.716858  0.746545  0.782428  0.914070  0.679870  0.805694  0.836966   
2  0.873693  0.776061  0.648815  0.798071  0.613755  0.963728  0.703512   
3  0.839160  0.968750  0.635397  0.678393  0.618091  0.730132  0.755471   
4  0

## Step 4: Build the PyTorch Geometric Graph

We construct a **single large graph** where:
- Each row in `features.csv` is a **node** (identified by `Vertex`)
- Each row in `edges.csv` is an **undirected edge**
- Each row in `labels.csv` is the **class label** for the corresponding node

In [18]:
# ── Node feature matrix x ───────────────────────────────────────────────────
feature_cols = [c for c in features_df.columns if c != 'Vertex']
features_sorted = features_df.sort_values('Vertex').reset_index(drop=True)
x = torch.tensor(features_sorted[feature_cols].values, dtype=torch.float)

# ── Node labels y ────────────────────────────────────────────────────────────
labels_sorted = labels_df.sort_values('Vertex').reset_index(drop=True)
y = torch.tensor(labels_sorted['label'].values, dtype=torch.long)

# ── Edge index (undirected) ───────────────────────────────────────────────────
src = torch.tensor(edges_df['Vertex 1'].values, dtype=torch.long)
dst = torch.tensor(edges_df['Vertex 2'].values, dtype=torch.long)
# Add both directions for an undirected graph
edge_index = torch.cat([
    torch.stack([src, dst], dim=0),
    torch.stack([dst, src], dim=0)
], dim=1)

# ── Create Data object ────────────────────────────────────────────────────────
graph_data = Data(x=x, edge_index=edge_index, y=y)

print(f"Graph summary:")
print(f"  Nodes          : {graph_data.num_nodes}")
print(f"  Edges (directed): {graph_data.num_edges}")
print(f"  Node features  : {graph_data.num_node_features}")
print(f"  Classes        : {y.unique().tolist()}")
print(f"  Is undirected  : {graph_data.is_undirected()}")


Graph summary:
  Nodes          : 2000
  Edges (directed): 16000
  Node features  : 16
  Classes        : [0, 1]
  Is undirected  : True


## Step 5: Create Train / Validation / Test Masks

We split **nodes** (not graphs) into 60 % train, 20 % validation, and 20 % test.

In [19]:
torch.manual_seed(42)
np.random.seed(42)

num_nodes  = graph_data.num_nodes
perm       = torch.randperm(num_nodes)

n_train = int(0.60 * num_nodes)
n_val   = int(0.20 * num_nodes)
# n_test  = num_nodes - n_train - n_val  (remainder)

train_idx = perm[:n_train]
val_idx   = perm[n_train : n_train + n_val]
test_idx  = perm[n_train + n_val :]

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask   = torch.zeros(num_nodes, dtype=torch.bool)
test_mask  = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

graph_data.train_mask = train_mask
graph_data.val_mask   = val_mask
graph_data.test_mask  = test_mask

print(f"Train nodes : {train_mask.sum().item()}")
print(f"Val nodes   : {val_mask.sum().item()}")
print(f"Test nodes  : {test_mask.sum().item()}")


Train nodes : 1200
Val nodes   : 400
Test nodes  : 400


## Step 6: Set Device

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
graph_data = graph_data.to(device)
print(f"Using device: {device}")

# Shared hyper-parameters
NUM_FEATURES = graph_data.num_node_features
NUM_CLASSES  = int(graph_data.y.max().item()) + 1
HIDDEN       = 64
EPOCHS       = 200
LR           = 0.01
WEIGHT_DECAY = 5e-4

print(f"Node features : {NUM_FEATURES}")
print(f"Num classes   : {NUM_CLASSES}")


Using device: cuda
Node features : 16
Num classes   : 2


## Step 7: Training and Evaluation Helpers

In [21]:
def train_epoch(model, optimizer, criterion, data):
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


def evaluate(model, data, mask):
    """Return (accuracy, precision, recall) on nodes selected by mask."""
    model.eval()
    with torch.no_grad():
        out  = model(data.x, data.edge_index)
        pred = out[mask].argmax(dim=1).cpu().numpy()
        true = data.y[mask].cpu().numpy()
    acc  = (pred == true).mean()
    prec = precision_score(true, pred, average='macro', zero_division=0)
    rec  = recall_score   (true, pred, average='macro', zero_division=0)
    return acc, prec, rec


def run_model(model, model_name, data, epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY):
    """Full training loop + final test evaluation."""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss()

    print(f"\n{'='*55}")
    print(f"  Training {model_name}")
    print(f"{'='*55}")

    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, optimizer, criterion, data)
        if epoch % 50 == 0:
            _, val_prec, val_rec = evaluate(model, data, data.val_mask)
            print(f"  Epoch {epoch:3d} | Loss: {loss:.4f} | "
                  f"Val Prec: {val_prec:.4f} | Val Rec: {val_rec:.4f}")

    # ── Final test evaluation ─────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        out  = model(data.x, data.edge_index)
        pred = out[data.test_mask].argmax(dim=1).cpu().numpy()
        true = data.y[data.test_mask].cpu().numpy()

    prec = precision_score(true, pred, average='macro', zero_division=0)
    rec  = recall_score   (true, pred, average='macro', zero_division=0)

    print(f"\n  ── {model_name} Test Results ──")
    print(f"     Precision (macro): {prec:.4f}")
    print(f"     Recall    (macro): {rec:.4f}")
    print("\n  Per-class report:")
    print(classification_report(true, pred, zero_division=0))

    return prec, rec


print("Helper functions defined.")


Helper functions defined.


## Step 8: Graph Convolutional Network (GCN)

Uses `GCNConv` — spectral-based convolution that averages neighbour features weighted by normalised degree.

In [22]:
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin   = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.lin(x)


torch.manual_seed(42)
gcn_model = GCN(NUM_FEATURES, HIDDEN, NUM_CLASSES)
print(gcn_model)
gcn_prec, gcn_rec = run_model(gcn_model, "GCN", graph_data)


GCN(
  (conv1): GCNConv(16, 64)
  (conv2): GCNConv(64, 64)
  (lin): Linear(in_features=64, out_features=2, bias=True)
)

  Training GCN
  Epoch  50 | Loss: 0.1018 | Val Prec: 0.9725 | Val Rec: 0.9725
  Epoch 100 | Loss: 0.0672 | Val Prec: 0.9801 | Val Rec: 0.9803
  Epoch 150 | Loss: 0.0595 | Val Prec: 0.9876 | Val Rec: 0.9877
  Epoch 200 | Loss: 0.0515 | Val Prec: 0.9851 | Val Rec: 0.9853

  ── GCN Test Results ──
     Precision (macro): 0.9900
     Recall    (macro): 0.9900

  Per-class report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       207
           1       0.99      0.99      0.99       193

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       0.99      0.99      0.99       400



## Step 9: Graph Attention Network (GAT)

Uses `GATConv` — attention mechanism computes a **weighted average** of neighbours based on learned attention coefficients.

In [23]:
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        # conv1 uses multi-head attention with concatenation → hidden_channels total
        self.conv1 = GATConv(in_channels,   hidden_channels // heads, heads=heads,
                              dropout=dropout, concat=True)
        # conv2 uses single head, average → hidden_channels
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1,
                              dropout=dropout, concat=False)
        self.lin   = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv2(x, edge_index))
        return self.lin(x)


torch.manual_seed(42)
gat_model = GAT(NUM_FEATURES, HIDDEN, NUM_CLASSES)
print(gat_model)
gat_prec, gat_rec = run_model(gat_model, "GAT", graph_data)


GAT(
  (conv1): GATConv(16, 16, heads=4)
  (conv2): GATConv(64, 64, heads=1)
  (lin): Linear(in_features=64, out_features=2, bias=True)
)

  Training GAT
  Epoch  50 | Loss: 0.1950 | Val Prec: 0.9950 | Val Rec: 0.9950
  Epoch 100 | Loss: 0.1856 | Val Prec: 0.9877 | Val Rec: 0.9873
  Epoch 150 | Loss: 0.1741 | Val Prec: 0.9877 | Val Rec: 0.9873
  Epoch 200 | Loss: 0.1815 | Val Prec: 0.9950 | Val Rec: 0.9950

  ── GAT Test Results ──
     Precision (macro): 0.9905
     Recall    (macro): 0.9896

  Per-class report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       207
           1       1.00      0.98      0.99       193

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       0.99      0.99      0.99       400



## Step 10: GraphSAGE

Uses `SAGEConv` — learns by **sampling and aggregating** features from local neighbourhoods, enabling inductive learning.

In [24]:
class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels,   hidden_channels, aggr='mean')
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, aggr='mean')
        self.lin   = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.lin(x)


torch.manual_seed(42)
sage_model = GraphSAGE(NUM_FEATURES, HIDDEN, NUM_CLASSES)
print(sage_model)
sage_prec, sage_rec = run_model(sage_model, "GraphSAGE", graph_data)


GraphSAGE(
  (conv1): SAGEConv(16, 64, aggr=mean)
  (conv2): SAGEConv(64, 64, aggr=mean)
  (lin): Linear(in_features=64, out_features=2, bias=True)
)

  Training GraphSAGE
  Epoch  50 | Loss: 0.0008 | Val Prec: 1.0000 | Val Rec: 1.0000
  Epoch 100 | Loss: 0.0004 | Val Prec: 1.0000 | Val Rec: 1.0000
  Epoch 150 | Loss: 0.0007 | Val Prec: 1.0000 | Val Rec: 1.0000
  Epoch 200 | Loss: 0.0004 | Val Prec: 1.0000 | Val Rec: 1.0000

  ── GraphSAGE Test Results ──
     Precision (macro): 1.0000
     Recall    (macro): 1.0000

  Per-class report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       207
           1       1.00      1.00      1.00       193

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



## Step 11: Summary Comparison

In [25]:
print("\n" + "="*55)
print("   FINAL RESULTS SUMMARY (Test Set — Macro Avg)")
print("="*55)
print(f"{'Model':<15} {'Precision':>12} {'Recall':>12}")
print("-"*55)
print(f"{'GCN':<15} {gcn_prec:>12.4f} {gcn_rec:>12.4f}")
print(f"{'GAT':<15} {gat_prec:>12.4f} {gat_rec:>12.4f}")
print(f"{'GraphSAGE':<15} {sage_prec:>12.4f} {sage_rec:>12.4f}")
print("="*55)



   FINAL RESULTS SUMMARY (Test Set — Macro Avg)
Model              Precision       Recall
-------------------------------------------------------
GCN                   0.9900       0.9900
GAT                   0.9905       0.9896
GraphSAGE             1.0000       1.0000
